In [1]:
# Qualitative AI Sentence Extraction (High-Precision Version)

import re
import html
import time
from pathlib import Path
from collections import Counter

import pandas as pd


# =========================
# USER SETTINGS
# =========================
DICT_FILE = r"/Volumes/ORICO/Dissertation/Hospitality_AI_Dictionary_v6_refined.xlsx"
DICT_SHEET = "Dictionary"
TEXT_FOLDER = r"/Volumes/ORICO/Dissertation/10-k"
OUTPUT_FILE = r"/Volumes/ORICO/Dissertation/AI_sentence_hits.xlsx"

MIN_WORDS_PER_FILE = 1000
SKIP_HIDDEN_FILES = True

EXTRACT_MAIN_10K_DOCUMENT = True
EXTRACT_KEY_SECTIONS_ONLY = False

# Qualitative extraction should prefer precision over recall.
MIN_SENT_WORDS = 8

# Remove short acronym terms that overmatch noisy SEC raw text.
EXCLUDE_SHORT_ACRONYMS = {
    "ai", "ocr", "nlp", "llm", "bert", "mlops"
}

# If you want to force a narrow qualitative dictionary, keep only terms
# that are longer-form or clearly interpretable in sentences.
QUALITATIVE_ALLOWLIST = {
    "artificial intelligence",
    "machine learning",
    "generative ai",
    "deep learning",
    "neural network",
    "computer vision",
    "facial recognition",
    "voice recognition",
    "speech recognition",
    "recommendation engine",
    "recommendation system",
    "chatbot",
    "virtual assistant",
    "service robot",
    "robot",
    "robotics",
    "robotic",
    "autonomous system",
    "intelligent agent",
    "personalization engine",
    "large language model",
    "transformer model",
    "conversational ai",
    "image recognition",
    "object recognition",
    "reinforcement learning",
}


# =========================
# FILE / DICTIONARY HELPERS
# =========================
def read_txt_file(file_path):
    for enc in ["utf-8", "utf-8-sig", "cp1252", "latin-1"]:
        try:
            return Path(file_path).read_text(encoding=enc, errors="ignore")
        except Exception:
            continue
    raise ValueError(f"Could not read file: {file_path}")


def load_terms(dict_file, sheet_name="Dictionary"):
    raw = pd.read_excel(dict_file, sheet_name=sheet_name, header=None)

    header_row = None
    for i in range(min(10, len(raw))):
        row_values = [
            str(x).strip().lower()
            for x in raw.iloc[i].tolist()
            if pd.notna(x)
        ]
        if "term" in row_values or "terms" in row_values:
            header_row = i
            break

    if header_row is None:
        raise ValueError("Dictionary header not found")

    df = pd.read_excel(dict_file, sheet_name=sheet_name, header=header_row)

    term_col = None
    for c in df.columns:
        if str(c).strip().lower() in ["term", "terms"]:
            term_col = c
            break

    if term_col is None:
        raise ValueError("Term column not found")

    terms = (
        df[term_col]
        .dropna()
        .astype(str)
        .str.strip()
        .tolist()
    )

    terms = list(dict.fromkeys(terms))
    return terms


def normalize_term(term):
    term = term.strip().lower()
    term = term.replace("_", " ")
    term = term.replace("/", " ")
    term = term.replace("–", "-").replace("—", "-")
    term = re.sub(r"(?i)\ba\.i\.\b", "ai", term)
    term = re.sub(r"(?i)\bartificial-intelligence\b", "artificial intelligence", term)
    term = re.sub(r"(?i)\bmachine-learning\b", "machine learning", term)
    term = re.sub(r"\s+", " ", term)
    return term


def filter_qualitative_terms(terms):
    cleaned = []
    for t in terms:
        nt = normalize_term(t)

        # Drop short acronym terms that are too noisy for sentence harvesting
        if nt in EXCLUDE_SHORT_ACRONYMS:
            continue

        # Keep only interpretable qualitative terms if allowlist is populated
        if QUALITATIVE_ALLOWLIST and nt not in QUALITATIVE_ALLOWLIST:
            continue

        cleaned.append(t)

    return list(dict.fromkeys(cleaned))


def build_term_patterns(terms):
    patterns = {}
    for term in terms:
        norm = normalize_term(term)
        escaped = re.escape(norm)
        escaped = escaped.replace(r"\ ", r"[\s\-]+")
        pattern = re.compile(rf"(?i)\b{escaped}\b")
        patterns[term] = pattern
    return patterns


def parse_firm_year(filename):
    stem = Path(filename).stem
    year_match = re.search(r"(19|20)\d{2}", stem)
    year = year_match.group(0) if year_match else None

    if year_match:
        firm = stem[:year_match.start()].strip("_- ")
    else:
        firm = re.split(r"[_\- ]+", stem)[0]

    return firm, year


# =========================
# CLEANING HELPERS
# =========================
def count_words(text):
    return len(re.findall(r"\b\w+\b", text))


def get_alpha_ratio(s):
    if not s:
        return 0.0
    alpha = sum(ch.isalpha() for ch in s)
    return alpha / max(len(s), 1)


def get_symbol_ratio(s):
    if not s:
        return 0.0
    symbols = sum(not (ch.isalnum() or ch.isspace()) for ch in s)
    return symbols / max(len(s), 1)


def get_vowel_ratio(s):
    letters = [ch.lower() for ch in s if ch.isalpha()]
    if not letters:
        return 0.0
    vowels = sum(ch in "aeiou" for ch in letters)
    return vowels / max(len(letters), 1)


def looks_like_encoded_blob(s):
    s = s.strip()
    if not s:
        return False

    if len(s) > 70 and get_alpha_ratio(s) < 0.35 and get_symbol_ratio(s) > 0.18:
        return True

    if len(s) > 70 and get_vowel_ratio(s) < 0.20:
        return True

    if len(s) > 60 and re.search(r"[A-Z0-9+/]{35,}={0,2}", s):
        return True

    if re.match(r"^M[^a-z]{10,}", s):
        return True

    if re.search(r"[`~^\\]{2,}", s) and len(s) > 50:
        return True

    return False


def extract_main_document(raw_text):
    docs = re.findall(r"(?is)<document>(.*?)</document>", raw_text)
    if not docs:
        return raw_text

    candidates = []
    for d in docs:
        type_match = re.search(r"(?is)<type>\s*([^\n<]+)", d)
        doc_type = type_match.group(1).strip().upper() if type_match else ""
        candidates.append((doc_type, d))

    preferred = ["10-K", "10K", "10-K405", "10K405", "ANNUAL REPORT"]
    for p in preferred:
        for doc_type, block in candidates:
            if doc_type.startswith(p):
                return block

    return max((block for _, block in candidates), key=len)


def strip_html_and_xbrl(text):
    text = html.unescape(text)

    text = re.sub(r"(?is)<script.*?>.*?</script>", " ", text)
    text = re.sub(r"(?is)<style.*?>.*?</style>", " ", text)
    text = re.sub(r"(?is)<!--.*?-->", " ", text)
    text = re.sub(r"(?is)</?(ix:|xbrli:|dei:|us-gaap:|xbrldi:|link:|xlink:)[^>]*>", " ", text)
    text = re.sub(r"(?is)<[^>]+>", " ", text)

    text = text.replace("\xa0", " ")
    text = text.replace("\u200b", " ")
    text = text.replace("–", "-").replace("—", "-")
    text = text.replace("_", " ")
    text = text.replace("/", " ")

    text = re.sub(r"(?i)\ba\.i\.\b", "ai", text)
    text = re.sub(r"(?i)\bartificial-intelligence\b", "artificial intelligence", text)
    text = re.sub(r"(?i)\bmachine-learning\b", "machine learning", text)

    return text


def filter_noisy_lines(text):
    lines = text.splitlines()
    kept = []

    for line in lines:
        s = line.strip()

        if not s:
            kept.append("")
            continue

        if len(s) < 5:
            continue

        if re.fullmatch(r"[-_=*]{3,}", s):
            continue

        if re.fullmatch(r"\d+", s):
            continue

        if re.fullmatch(r"page\s+\d+", s.lower()):
            continue

        if looks_like_encoded_blob(s):
            continue

        if len(s) > 50 and get_alpha_ratio(s) < 0.45:
            continue

        if len(s) > 50 and get_vowel_ratio(s) < 0.22:
            continue

        kept.append(s)

    text = "\n".join(kept)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s*\n+", "\n\n", text)
    return text.strip()


def extract_key_sections(text):
    section_specs = [
        ("item1_business",
         r"(?im)^\s*item\s+1\s*[\.\-:]?\s*business\b",
         [r"(?im)^\s*item\s+1a\s*[\.\-:]?\s*risk\s+factors\b",
          r"(?im)^\s*item\s+2\s*[\.\-:]?\b"]),
        ("item1a_risk_factors",
         r"(?im)^\s*item\s+1a\s*[\.\-:]?\s*risk\s+factors\b",
         [r"(?im)^\s*item\s+1b\s*[\.\-:]?\b",
          r"(?im)^\s*item\s+2\s*[\.\-:]?\b"]),
        ("item7_mda",
         r"(?im)^\s*item\s+7\s*[\.\-:]?\s*(management[’'`s]{0,2}\s+discussion\s+and\s+analysis|md&a)\b",
         [r"(?im)^\s*item\s+7a\s*[\.\-:]?\b",
          r"(?im)^\s*item\s+8\s*[\.\-:]?\s*financial\s+statements\b"])
    ]

    parts = []
    for _, start_pat, end_pats in section_specs:
        m = re.search(start_pat, text, flags=re.I | re.M)
        if not m:
            continue
        start_idx = m.start()
        search_after = text[m.end():]
        end_candidates = []
        for ep in end_pats:
            em = re.search(ep, search_after, flags=re.I | re.M)
            if em:
                end_candidates.append(m.end() + em.start())
        end_idx = min(end_candidates) if end_candidates else len(text)
        section = text[start_idx:end_idx].strip()
        if section:
            parts.append(section)

    if parts:
        return "\n\n".join(parts)
    return text


def normalize_text(raw_text):
    text = raw_text

    if EXTRACT_MAIN_10K_DOCUMENT:
        text = extract_main_document(text)

    text = strip_html_and_xbrl(text)
    text = filter_noisy_lines(text)

    if EXTRACT_KEY_SECTIONS_ONLY:
        text = extract_key_sections(text)

    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s*\n+", "\n\n", text)
    text = re.sub(r"\s+\.", ".", text)

    return text.strip()


def sentence_quality_ok(s):
    if len(s.split()) < MIN_SENT_WORDS:
        return False

    if len(s) > 60 and get_alpha_ratio(s) < 0.65:
        return False

    if len(s) > 60 and get_symbol_ratio(s) > 0.12:
        return False

    if len(s) > 60 and get_vowel_ratio(s) < 0.24:
        return False

    if looks_like_encoded_blob(s):
        return False

    return True


def simple_sentence_split(text):
    text = text.replace("\r", "\n")
    text = re.sub(r"\n+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    sentences = re.split(
        r'(?<=[\.\?\!])\s+(?=["“”\'(\[]?[A-Z])',
        text
    )

    cleaned = []
    for s in sentences:
        s = s.strip()
        if not s:
            continue
        if sentence_quality_ok(s):
            cleaned.append(s)

    return cleaned


# =========================
# MAIN
# =========================
def main():
    run_start = time.perf_counter()

    all_terms = load_terms(DICT_FILE, DICT_SHEET)
    terms = filter_qualitative_terms(all_terms)
    patterns = build_term_patterns(terms)

    sentence_hits = []
    term_hits = []
    file_summary = []

    files = sorted(Path(TEXT_FOLDER).glob("*.txt"))

    for idx, file_path in enumerate(files, start=1):
        file_start = time.perf_counter()
        filename = file_path.name

        if SKIP_HIDDEN_FILES and filename.startswith("._"):
            continue

        firm, year = parse_firm_year(filename)

        try:
            raw = read_txt_file(file_path)
            raw_words = count_words(raw)

            cleaned = normalize_text(raw)
            cleaned_words = count_words(cleaned)

            if cleaned_words < MIN_WORDS_PER_FILE:
                file_summary.append({
                    "file_name": filename,
                    "firm": firm,
                    "year": year,
                    "raw_word_count": raw_words,
                    "cleaned_word_count": cleaned_words,
                    "sentence_count": 0,
                    "matched_sentence_count": 0,
                    "matched_term_count": 0,
                    "top_terms": "",
                    "status": "skipped_too_short_after_cleaning",
                    "elapsed_seconds": round(time.perf_counter() - file_start, 2),
                })
                continue

            sentences = simple_sentence_split(cleaned)
            matched_sentence_count = 0
            matched_term_counter = Counter()

            for i, sent in enumerate(sentences):
                matched_terms = []

                for term, pattern in patterns.items():
                    if pattern.search(sent):
                        matched_terms.append(term)

                if matched_terms:
                    matched_sentence_count += 1

                    prev_sent = sentences[i - 1] if i > 0 else ""
                    next_sent = sentences[i + 1] if i < len(sentences) - 1 else ""
                    sid = f"{filename}__{i}"
                    uniq_terms = sorted(set(matched_terms))

                    sentence_hits.append({
                        "sentence_id": sid,
                        "file_name": filename,
                        "firm": firm,
                        "year": year,
                        "matched_term_count": len(uniq_terms),
                        "matched_terms": "; ".join(uniq_terms),
                        "previous_sentence": prev_sent,
                        "sentence": sent,
                        "next_sentence": next_sent,
                    })

                    for mt in uniq_terms:
                        matched_term_counter[mt] += 1
                        term_hits.append({
                            "sentence_id": sid,
                            "file_name": filename,
                            "firm": firm,
                            "year": year,
                            "matched_term": mt,
                            "sentence": sent,
                        })

            elapsed = round(time.perf_counter() - file_start, 2)

            file_summary.append({
                "file_name": filename,
                "firm": firm,
                "year": year,
                "raw_word_count": raw_words,
                "cleaned_word_count": cleaned_words,
                "sentence_count": len(sentences),
                "matched_sentence_count": matched_sentence_count,
                "matched_term_count": int(sum(matched_term_counter.values())),
                "top_terms": "; ".join([f"{k}:{v}" for k, v in matched_term_counter.most_common(10)]),
                "status": "ok",
                "elapsed_seconds": elapsed,
            })

            print(f"[{idx}/{len(files)}] OK - {filename} - {elapsed}s")

        except Exception as e:
            elapsed = round(time.perf_counter() - file_start, 2)
            file_summary.append({
                "file_name": filename,
                "firm": firm,
                "year": year,
                "raw_word_count": None,
                "cleaned_word_count": None,
                "sentence_count": None,
                "matched_sentence_count": None,
                "matched_term_count": None,
                "top_terms": "",
                "status": f"error: {str(e)}",
                "elapsed_seconds": elapsed,
            })
            print(f"[{idx}/{len(files)}] ERROR - {filename} - {elapsed}s - {e}")

    sentence_df = pd.DataFrame(sentence_hits)
    term_df = pd.DataFrame(term_hits)
    summary_df = pd.DataFrame(file_summary)

    if not sentence_df.empty:
        sentence_df = sentence_df.sort_values(
            ["firm", "year", "file_name", "sentence_id"]
        ).reset_index(drop=True)

    if not term_df.empty:
        term_df = term_df.sort_values(
            ["firm", "year", "file_name", "matched_term"]
        ).reset_index(drop=True)

    if not summary_df.empty:
        summary_df = summary_df.sort_values(
            ["status", "firm", "year", "file_name"]
        ).reset_index(drop=True)

    total_elapsed = round(time.perf_counter() - run_start, 2)
    config_df = pd.DataFrame([{
        "DICT_FILE": DICT_FILE,
        "DICT_SHEET": DICT_SHEET,
        "TEXT_FOLDER": TEXT_FOLDER,
        "OUTPUT_FILE": OUTPUT_FILE,
        "MIN_WORDS_PER_FILE": MIN_WORDS_PER_FILE,
        "SKIP_HIDDEN_FILES": SKIP_HIDDEN_FILES,
        "EXTRACT_MAIN_10K_DOCUMENT": EXTRACT_MAIN_10K_DOCUMENT,
        "EXTRACT_KEY_SECTIONS_ONLY": EXTRACT_KEY_SECTIONS_ONLY,
        "MIN_SENT_WORDS": MIN_SENT_WORDS,
        "TOTAL_ELAPSED_SECONDS": total_elapsed,
        "TOTAL_FILES_FOUND": len(files),
        "TOTAL_TERMS_AFTER_QUAL_FILTER": len(terms),
        "TOTAL_SENTENCE_HITS": len(sentence_df),
        "TOTAL_TERM_HITS": len(term_df)
    }])

    with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
        sentence_df.to_excel(writer, index=False, sheet_name="Sentence_Hits")
        term_df.to_excel(writer, index=False, sheet_name="Term_Hits")
        summary_df.to_excel(writer, index=False, sheet_name="File_Summary")
        config_df.to_excel(writer, index=False, sheet_name="Config")

    print("Finished.")
    print(f"Saved to: {OUTPUT_FILE}")
    print(f"Sentence hits: {len(sentence_df)}")
    print(f"Term hits: {len(term_df)}")
    print(f"Total elapsed seconds: {total_elapsed}")


main()


[199/1402] OK - 0083A_2010.txt - 3.47s
[200/1402] OK - 0083A_2011.txt - 9.13s
[201/1402] OK - 0083A_2012.txt - 25.71s
[202/1402] OK - 0182A_2011.txt - 5.66s
[203/1402] OK - 0182A_2012.txt - 9.46s
[204/1402] OK - 0182A_2013.txt - 9.41s
[205/1402] OK - 0182A_2014.txt - 10.55s
[206/1402] OK - 0738B_2012.txt - 15.33s
[207/1402] OK - 0738B_2013.txt - 15.49s
[208/1402] OK - 0738B_2014.txt - 14.73s
[209/1402] OK - 0738B_2015.txt - 11.65s
[210/1402] OK - 0738B_2016.txt - 10.61s
[211/1402] OK - 2638B_2010.txt - 0.22s
[212/1402] OK - 2638B_2011.txt - 1.48s
[213/1402] OK - 2638B_2012.txt - 4.3s
[214/1402] OK - 2638B_2013.txt - 4.29s
[215/1402] OK - 2638B_2014.txt - 3.83s
[216/1402] OK - 2638B_2015.txt - 3.17s
[217/1402] OK - 2638B_2016.txt - 3.05s
[218/1402] OK - 2638B_2017.txt - 2.91s
[219/1402] OK - 2638B_2018.txt - 3.17s
[220/1402] OK - 2638B_2019.txt - 3.5s
[221/1402] OK - 2638B_2020.txt - 3.49s
[222/1402] OK - 2638B_2021.txt - 3.79s
[223/1402] OK - 2638B_2022.txt - 3.85s
[224/1402] OK - 2638

In [4]:
import pandas as pd

result_file = r"/Volumes/ORICO/Dissertation/AI_sentence_hits.xlsx"

sentence_df = pd.read_excel(result_file, sheet_name="Sentence_Hits")
term_df = pd.read_excel(result_file, sheet_name="Term_Hits")
summary_df = pd.read_excel(result_file, sheet_name="File_Summary")
config_df = pd.read_excel(result_file, sheet_name="Config")

print(sentence_df.shape)
print(term_df.shape)
print(summary_df.shape)
print(config_df.shape)

(343, 9)
(368, 6)
(1204, 11)
(1, 14)


In [5]:
term_df["matched_term"].value_counts().head(30)

matched_term
artificial intelligence    202
Machine learning            72
Robotic                     35
Generative AI               31
Chatbot                     13
Facial recognition          10
Voice recognition            2
Robot                        1
Recommendation engine        1
Robotics                     1
Name: count, dtype: int64